# NB03 — Baselines

**GraphSentry: A Unified GNN Framework for Blockchain Illicit Activity Detection**

This notebook trains baseline models for comparative evaluation against GraphSentry
(NB02). All baselines use a standard DataLoader with oversampled minority class,
the same train/val/test split, and the same evaluation protocol.

**Baselines:**
1. **Subgraph GCN (DataLoader)** — same architecture as GraphSentry but without
   GraphSAINT sampling. Isolates the sampling contribution.
2. **Subgraph GAT** — Graph Attention Network with multi-head attention, subgraph-level.
3. **Subgraph GraphSAGE** — GraphSAGE with mean aggregation, subgraph-level.

**Requires:** Artefacts from `NB01_data_pipeline.ipynb`

**Produces:**
- `baseline_results.json` — metrics for all baselines

## 1. Configuration

In [28]:
MAX_EPOCHS = 60
PATIENCE = 10
BATCH_SIZE = 64
LR = 0.005
LR_FACTOR = 0.5
LR_PATIENCE = 5
HIDDEN_DIM = 128
DROPOUT = 0.5
SEED = 42

BASE_PATH = '/content/drive/MyDrive/GraphSentry'
PROCESSED_PATH = f'{BASE_PATH}/data/processed'

## 2. Environment + load artefacts

In [29]:
!pip install -q uv
!uv pip install --system torch torch-geometric numpy pandas tqdm scikit-learn

from google.colab import drive
import os, json, time, copy

import torch
import torch.nn.functional as F
import numpy as np
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, GATConv, SAGEConv, global_mean_pool
from torch_geometric.utils import degree
from torch_geometric.loader import DataLoader
from sklearn.metrics import (
    f1_score, roc_auc_score, precision_score, recall_score, confusion_matrix
)

np.random.seed(SEED)
torch.manual_seed(SEED)

drive.mount('/content/drive')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

bg = torch.load(os.path.join(PROCESSED_PATH, 'background_graph.pt'), weights_only=False)
meta = torch.load(os.path.join(PROCESSED_PATH, 'cc_metadata.pt'), weights_only=False)
feat_anon = torch.load(os.path.join(PROCESSED_PATH, 'features_anonymous.pt'), weights_only=False)

with open(os.path.join(PROCESSED_PATH, 'dataset_stats.json'), 'r') as f:
    stats = json.load(f)

INPUT_DIM = stats['feature_dims']['anonymous'] + 1
print(f"Dataset: {stats['total_ccs']} CCs, {stats['total_nodes']} nodes, input dim = {INPUT_DIM}")

Using Python 3.12.13 environment at: /usr
Checked 6 packages in 99ms
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
Dataset: 5500 CCs, 39819 nodes, input dim = 44


## 3. Build datasets

In [30]:
def build_pyg_graph(cc_id):
    cc_nodes = sorted(bg['cc_to_nodes'][cc_id])
    local_map = {g: l for l, g in enumerate(cc_nodes)}
    adj = bg['adj_list']

    x = feat_anon[cc_nodes].clone()

    node_set = set(cc_nodes)
    local_src, local_dst = [], []
    for g_src in cc_nodes:
        for g_dst in adj[g_src]:
            if g_dst in node_set and g_dst in local_map:
                local_src.append(local_map[g_src])
                local_dst.append(local_map[g_dst])

    if len(local_src) == 0:
        edge_index = torch.tensor([[0], [0]], dtype=torch.long)
    else:
        edge_index = torch.tensor([local_src, local_dst], dtype=torch.long)

    deg = degree(edge_index[1], x.size(0), dtype=torch.float)
    deg = torch.log(deg + 1).view(-1, 1)
    x = torch.cat([x, deg], dim=1)

    y = torch.tensor([meta['cc_label_map'][cc_id]], dtype=torch.long)
    return Data(x=x, edge_index=edge_index, y=y)


train_data = [build_pyg_graph(cc_id) for cc_id in meta['train_ids']]
val_data = [build_pyg_graph(cc_id) for cc_id in meta['val_ids']]
test_data = [build_pyg_graph(cc_id) for cc_id in meta['test_ids']]

train_pos = [d for d in train_data if d.y.item() == 1]
train_neg = [d for d in train_data if d.y.item() == 0]
oversample_factor = max(1, len(train_neg) // len(train_pos))
train_balanced = train_neg + train_pos * oversample_factor

train_loader = DataLoader(train_balanced, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE)

print(f"Train: {len(train_balanced)} (oversampled from {len(train_data)})")
print(f"Val:   {len(val_data)}")
print(f"Test:  {len(test_data)}")

Train: 7000 (oversampled from 3850)
Val:   825
Test:  825


## 4. Model architectures

In [31]:
class SubgraphGCN(torch.nn.Module):
    def __init__(self, in_channels, hidden=HIDDEN_DIM, dropout=DROPOUT):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden)
        self.bn1 = torch.nn.BatchNorm1d(hidden)
        self.conv2 = GCNConv(hidden, hidden)
        self.bn2 = torch.nn.BatchNorm1d(hidden)
        self.conv3 = GCNConv(hidden, hidden)
        self.bn3 = torch.nn.BatchNorm1d(hidden)
        self.classifier = torch.nn.Linear(hidden, 2)
        self.dropout = dropout

    def forward(self, x, edge_index, batch):
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = F.relu(self.bn2(self.conv2(x, edge_index)))
        x = self.bn3(self.conv3(x, edge_index))
        x = global_mean_pool(x, batch)
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.classifier(x)


class SubgraphGAT(torch.nn.Module):
    def __init__(self, in_channels, hidden=HIDDEN_DIM, heads=4, dropout=DROPOUT):
        super().__init__()
        self.conv1 = GATConv(in_channels, hidden // heads, heads=heads)
        self.bn1 = torch.nn.BatchNorm1d(hidden)
        self.conv2 = GATConv(hidden, hidden // heads, heads=heads)
        self.bn2 = torch.nn.BatchNorm1d(hidden)
        self.conv3 = GATConv(hidden, hidden, heads=1)
        self.bn3 = torch.nn.BatchNorm1d(hidden)
        self.classifier = torch.nn.Linear(hidden, 2)
        self.dropout = dropout

    def forward(self, x, edge_index, batch):
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = F.relu(self.bn2(self.conv2(x, edge_index)))
        x = self.bn3(self.conv3(x, edge_index))
        x = global_mean_pool(x, batch)
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.classifier(x)


class SubgraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden=HIDDEN_DIM, dropout=DROPOUT):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden)
        self.bn1 = torch.nn.BatchNorm1d(hidden)
        self.conv2 = SAGEConv(hidden, hidden)
        self.bn2 = torch.nn.BatchNorm1d(hidden)
        self.conv3 = SAGEConv(hidden, hidden)
        self.bn3 = torch.nn.BatchNorm1d(hidden)
        self.classifier = torch.nn.Linear(hidden, 2)
        self.dropout = dropout

    def forward(self, x, edge_index, batch):
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = F.relu(self.bn2(self.conv2(x, edge_index)))
        x = self.bn3(self.conv3(x, edge_index))
        x = global_mean_pool(x, batch)
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.classifier(x)


for name, cls in [('GCN', SubgraphGCN), ('GAT', SubgraphGAT), ('SAGE', SubgraphSAGE)]:
    m = cls(INPUT_DIM)
    p = sum(p.numel() for p in m.parameters())
    print(f"{name:5s}: {p:,} parameters")

GCN  : 39,810 parameters
GAT  : 40,578 parameters
SAGE : 78,210 parameters


## 5. Training and evaluation functions

In [32]:
def evaluate(model, loader):
    model.eval()
    y_true, y_prob = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch.x, batch.edge_index, batch.batch)
            prob = F.softmax(out, dim=1)[:, 1]
            y_true.extend(batch.y.cpu().numpy())
            y_prob.extend(prob.cpu().numpy())

    y_true = np.array(y_true)
    y_prob = np.array(y_prob)

    has_both = len(np.unique(y_true)) > 1
    return {
        'auroc': roc_auc_score(y_true, y_prob) if has_both else 0.0,
        'f1': f1_score(y_true, (y_prob >= 0.5).astype(int), zero_division=0),
        'y_true': y_true,
        'y_prob': y_prob,
    }


def tune_threshold(y_true, y_prob):
    best_f1, best_t = 0, 0.5
    for t in np.arange(0.1, 1.0, 0.05):
        f = f1_score(y_true, (y_prob >= t).astype(int), zero_division=0)
        if f > best_f1:
            best_f1, best_t = f, t
    return best_t, best_f1


def train_and_evaluate(model_cls, name):
    torch.manual_seed(SEED)
    model = model_cls(INPUT_DIM).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=LR_FACTOR, patience=LR_PATIENCE
    )

    best_val_auroc = 0
    best_state = None
    no_improve = 0
    history = []

    t0 = time.time()
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        epoch_loss = 0
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out = model(batch.x, batch.edge_index, batch.batch)
            loss = F.cross_entropy(out, batch.y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(train_loader)
        val_m = evaluate(model, val_loader)
        scheduler.step(val_m['auroc'])

        if val_m['auroc'] > best_val_auroc:
            best_val_auroc = val_m['auroc']
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
            marker = ' *'
        else:
            no_improve += 1
            marker = ''

        history.append({
            'epoch': epoch, 'loss': avg_loss,
            'val_auroc': val_m['auroc'], 'val_f1': val_m['f1'],
        })

        if epoch <= 3 or epoch % 10 == 0 or marker:
            print(f"  {name:5s} Epoch {epoch:3d} | Loss: {avg_loss:.4f} | "
                  f"Val AUROC: {val_m['auroc']:.4f} | Val F1: {val_m['f1']:.4f}{marker}")

        if no_improve >= PATIENCE:
            print(f"  {name:5s} Early stopping at epoch {epoch}")
            break

    t_elapsed = time.time() - t0
    model.load_state_dict(best_state)

    val_r = evaluate(model, val_loader)
    best_thresh, val_f1_tuned = tune_threshold(val_r['y_true'], val_r['y_prob'])

    test_r = evaluate(model, test_loader)
    y_pred = (test_r['y_prob'] >= best_thresh).astype(int)
    cm = confusion_matrix(test_r['y_true'], y_pred)

    result = {
        'name': name,
        'best_val_auroc': round(best_val_auroc, 4),
        'threshold': best_thresh,
        'test_auroc': round(test_r['auroc'], 4),
        'test_f1': round(f1_score(test_r['y_true'], y_pred, zero_division=0), 4),
        'test_precision': round(precision_score(test_r['y_true'], y_pred, zero_division=0), 4),
        'test_recall': round(recall_score(test_r['y_true'], y_pred, zero_division=0), 4),
        'confusion_matrix': {'tn': int(cm[0,0]), 'fp': int(cm[0,1]),
                             'fn': int(cm[1,0]), 'tp': int(cm[1,1])},
        'epochs_run': len(history),
        'training_time_s': round(t_elapsed, 1),
    }

    print(f"  {name:5s} => Test AUROC: {result['test_auroc']:.4f} | "
          f"F1: {result['test_f1']:.4f} @ thresh={best_thresh:.2f} | "
          f"Time: {t_elapsed:.1f}s\n")

    return result

## 6. Run baselines

In [33]:
all_results = {}

print("=" * 70)
print("Baseline 1: Subgraph GCN (DataLoader, no GraphSAINT)")
print("=" * 70)
all_results['gcn_dataloader'] = train_and_evaluate(SubgraphGCN, 'GCN')

print("=" * 70)
print("Baseline 2: Subgraph GAT (4-head attention)")
print("=" * 70)
all_results['gat'] = train_and_evaluate(SubgraphGAT, 'GAT')

print("=" * 70)
print("Baseline 3: Subgraph GraphSAGE (mean aggregation)")
print("=" * 70)
all_results['sage'] = train_and_evaluate(SubgraphSAGE, 'SAGE')

Baseline 1: Subgraph GCN (DataLoader, no GraphSAINT)
  GCN   Epoch   1 | Loss: 0.5386 | Val AUROC: 0.8755 | Val F1: 0.3734 *
  GCN   Epoch   2 | Loss: 0.4409 | Val AUROC: 0.8616 | Val F1: 0.3758
  GCN   Epoch   3 | Loss: 0.3770 | Val AUROC: 0.8508 | Val F1: 0.3671
  GCN   Epoch  10 | Loss: 0.1815 | Val AUROC: 0.8434 | Val F1: 0.4021
  GCN   Early stopping at epoch 11
  GCN   => Test AUROC: 0.8526 | F1: 0.4105 @ thresh=0.75 | Time: 11.7s

Baseline 2: Subgraph GAT (4-head attention)
  GAT   Epoch   1 | Loss: 0.5197 | Val AUROC: 0.9024 | Val F1: 0.4012 *
  GAT   Epoch   2 | Loss: 0.4000 | Val AUROC: 0.8888 | Val F1: 0.4025
  GAT   Epoch   3 | Loss: 0.3405 | Val AUROC: 0.9073 | Val F1: 0.4553 *
  GAT   Epoch   4 | Loss: 0.2789 | Val AUROC: 0.9210 | Val F1: 0.5061 *
  GAT   Epoch  10 | Loss: 0.1575 | Val AUROC: 0.8974 | Val F1: 0.4817
  GAT   Early stopping at epoch 14
  GAT   => Test AUROC: 0.8930 | F1: 0.4747 @ thresh=0.70 | Time: 18.4s

Baseline 3: Subgraph GraphSAGE (mean aggregation)
 

## 7. Comparison table

In [34]:
with open(os.path.join(PROCESSED_PATH, 'training_results.json'), 'r') as f:
    graphsentry = json.load(f)

print("=" * 80)
print("COMPARISON: GraphSentry vs Baselines")
print("=" * 80)
print(f"{'Model':<35s} {'AUROC':>8s} {'F1':>8s} {'Prec':>8s} {'Recall':>8s} {'Thresh':>8s}")
print("-" * 80)

gs = graphsentry['test_metrics']
print(f"{'GraphSentry (GCN + GraphSAINT)':<35s} "
      f"{gs['auroc']:>8.4f} {gs['f1']:>8.4f} {gs['precision']:>8.4f} "
      f"{gs['recall']:>8.4f} {graphsentry['threshold']:>8.2f}")

for key in ['gcn_dataloader', 'gat', 'sage']:
    r = all_results[key]
    print(f"{r['name'] + ' (DataLoader)':<35s} "
          f"{r['test_auroc']:>8.4f} {r['test_f1']:>8.4f} {r['test_precision']:>8.4f} "
          f"{r['test_recall']:>8.4f} {r['threshold']:>8.2f}")

print("-" * 80)

COMPARISON: GraphSentry vs Baselines
Model                                  AUROC       F1     Prec   Recall   Thresh
--------------------------------------------------------------------------------
GraphSentry (GCN + GraphSAINT)        0.8931   0.4577   0.3651   0.6133     0.75
GCN (DataLoader)                      0.8526   0.4105   0.3052   0.6267     0.75
GAT (DataLoader)                      0.8930   0.4747   0.3821   0.6267     0.70
SAGE (DataLoader)                     0.8663   0.4431   0.4022   0.4933     0.90
--------------------------------------------------------------------------------


## 8. Save results

In [35]:
with open(os.path.join(PROCESSED_PATH, 'baseline_results.json'), 'w') as f:
    json.dump(all_results, f, indent=2)

print("Saved baseline_results.json")
print("\nReady for NB04.")

Saved baseline_results.json

Ready for NB04.
